# v9 Task 1 — regime characterization: earn the bandwidth verdict (root/bare-metal T4)

**Not a new kernel — a measurement.** Since v6 every decode step read "~10% HBM, per-CTA-bound." The v8
close-out (C12) showed that's **confounded**: the bench KV fit in the T4's 4 MB L2 (so the DRAM counter
reads low even if the kernel is memory-bound) and free Colab never locked clocks. v9 Task 2 (FP8) then
*partly* refuted it — FP8 bought a real ~1.3× **L2-load-bandwidth** win — but the limiter *name* is still
open.

**This notebook settles it:** lock clocks, **flush L2 between iters**, and sweep N_k 1K→128K so the KV
working set crosses 4 MB, measuring achieved **%HBM / effective-BW** + the **counter-free L2 test**
(`eff_bw > HBM peak ⇒ data came from L2`). The decisive plot is **%HBM vs N_k**:
- climbs toward the ceiling as N_k passes L2 → **genuinely memory-bound** (FP8's byte win vindicated);
- stays ~10% past L2 with `L2served=False` → **per-CTA-bound, confound-free at last**.

**Needs a root T4** (vast.ai) to lock clocks + run ncu; on free Colab it runs unlocked (counter-free
signal survives, cross-run wall-times don't). Both outcomes are first-class results.

## 0. Dependencies + GPU (venv-safe; adds matplotlib)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU on this runtime. FIX: pick a T4 (Colab) or a root/bare-metal T4 (vast.ai).')

# matplotlib for the decisive plot (the one new dep vs other gates); numpy before torch.
pip('ninja', 'pytest', 'numpy', 'matplotlib')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit('A GPU is present but torch was CPU-only -- installed CUDA build. Restart + re-run.')

os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap,clocks.current.sm,clocks.max.sm --format=csv

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Lock clocks (root only — loud warning + continue if not)

In [ ]:
# Lock clocks so wall-times are comparable across runs. NEEDS ROOT (vast.ai bare-metal). On free
# Colab this prints a LOUD warning and continues — the counter-free %HBM/eff_bw sweep still runs, but
# cross-run wall-times are confounded (exactly the C12 problem). Reset is in the last cell.
from bench.regime import lock_clocks
ok, sm_mhz, mem_mhz, throttle = lock_clocks()
print('locked =', ok, '| sm =', sm_mhz, 'MHz | mem =', mem_mhz, 'MHz | throttle:', throttle or 'none')
if not ok:
    print('\\n>>> Running UNLOCKED: trust %HBM / eff_bw / L2! (intra-run, clock-robust);')
    print('>>> do NOT compare absolute us/tok across runs. For the publishable verdict, use a root T4.')

## 3. Roofline framing — record the prediction + the L2-crossing N_k

In [ ]:
# Roofline framing (record BEFORE the run). Decode is AI = 2/b, HBM-bound at every N_k per the
# model -- BUT the model assumes all traffic hits HBM; it is BLIND to L2. Task 1 tests that empirically.
from roofline.archs import get_arch
arch = get_arch('sm_75')
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s | L2', arch.l2_mb, 'MB')
print('\\nL2-capacity crossing (KV working set = 2*N_k*d*b = 4 MB), B=1 H_kv=1:')
for d in (64, 128):
    for b, lab in ((2, 'fp16'), (1, 'fp8')):
        n_cross = int(arch.l2_mb*1e6 / (2*d*b))
        print(f'  d={d:3d} {lab}: N_k ~= {n_cross:6d}  (KV exceeds L2 past this)')
print('\\nPREDICTION: if memory-bound, %HBM stays ~flat (low) while WS <= 4 MB (L2-resident), then')
print('CLIMBS toward the achievable ceiling (~65-75%% of 320 GB/s) once N_k pushes KV past L2.')
print('COUNTER (the C12 survivor): if %HBM stays ~10%% past L2 with L2served=False -> per-CTA-bound,')
print('confound-free. FP8 halves the working set, so its climb (if any) starts at ~2x the N_k.')

## 4. Build v8_gqa_ss (FP16) + v9_fp8 (FP8)

In [ ]:
import glob, os, shutil
from bindings.load import build_kernel
for name in ('v8_gqa_ss', 'v9_fp8'):
    for d in glob.glob(os.path.expanduser(f'~/.cache/torch_extensions/*/fa_{name}')):
        if not glob.glob(os.path.join(d, '*.so')):
            shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
ss  = build_kernel('v8_gqa_ss'); print('built v8_gqa_ss:', ss is not None)
fp8 = build_kernel('v9_fp8');    print('built v9_fp8:',   fp8 is not None)

## 5. THE ISOLATION SWEEP — N_k 1K→128K, L2-flushed (returns rows for the plot)

In [ ]:
# THE ISOLATION SWEEP: B=1, H_kv in {1,8}, d in {64,128}, N_k 1K..128K, both kernels, L2 FLUSHED.
# H_kv=1 is the cleanest crossing (KV = 2*N_k*d*b). Returns structured rows we plot next.
from bench.regime import sweep
KV = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
print('================ v8_gqa_ss (FP16 KV) ================')
rows_ss  = sweep('v8_gqa_ss', KV, batches=[1], head_dims=[64, 128], h_kvs=[1, 8])
print('\\n================ v9_fp8 (FP8 KV) ====================')
rows_fp8 = sweep('v9_fp8',    KV, batches=[1], head_dims=[64, 128], h_kvs=[1, 8])
all_rows = rows_ss + rows_fp8
print('\\ncollected', len(all_rows), 'rows')

## 6. THE DECISIVE PLOT — %HBM vs N_k (does it climb past L2?)

In [ ]:
# THE DECISIVE PLOT: %HBM vs N_k (log-x), one line per (backend, d) at H_kv=1 (the clean isolation).
# Mark the L2-capacity crossing (KV = 4 MB), the achievable ceiling (~70%% of peak), and annotate any
# row where the counter-free test fired (L2served = eff_bw > HBM peak -> %HBM is meaningless there).
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for backend, rows, ls in (('v8_gqa_ss', rows_ss, '-'), ('v9_fp8', rows_fp8, '--')):
    for d, color in ((64, 'tab:blue'), (128, 'tab:red')):
        pts = sorted([r for r in rows if r['H_kv'] == 1 and r['d'] == d], key=lambda r: r['N_k'])
        if not pts:
            continue
        xs = [r['N_k'] for r in pts]; ys = [r['hbm_pct'] for r in pts]
        ax.plot(xs, ys, ls, marker='o', color=color, label=f'{backend} d={d}')
        for r in pts:
            if r['l2_served']:
                ax.annotate('L2!', (r['N_k'], r['hbm_pct']), fontsize=7, color='green')
ax.axhline(70, color='gray', ls=':', lw=1, label='achievable ceiling (~70% of peak)')
# L2 crossing for d=128 fp16 (the reference isolation): N_k where 2*N_k*128*2 = 4 MB
ax.axvline(int(arch.l2_mb*1e6/(2*128*2)), color='black', ls=':', lw=1, label='L2 crossing (d128 fp16)')
ax.set_xscale('log', base=2); ax.set_xlabel('N_k (KV length)'); ax.set_ylabel('% of peak HBM BW')
ax.set_title('v9 Task 1 — decode %HBM vs N_k (B=1, H_kv=1, L2-flushed)')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
os.makedirs('docs/diagrams', exist_ok=True)
fig.savefig('docs/diagrams/v9-task1-regime.svg', bbox_inches='tight')
fig.savefig('docs/diagrams/v9-task1-regime.png', dpi=110, bbox_inches='tight')
print('saved docs/diagrams/v9-task1-regime.svg')
plt.show()

## 7. Large-batch confirmation — N_k past L2, sweep B (climb or flat?)

In [ ]:
# LARGE-BATCH CONFIRMATION: fix N_k PAST L2 (16384 at H_kv=1, KV ~= 8 MB > 4 MB L2), sweep B.
# Does %HBM climb as BH passes ~2*SM (80 on T4), or stay flat? Flat-past-L2 = per-CTA-bound confirmed.
print('================ v8_gqa_ss, N_k=16384 (past L2), batch sweep ================')
_ = sweep('v8_gqa_ss', [16384], batches=[1, 8, 32, 64, 128], head_dims=[128], h_kvs=[1], max_ws_gb=16.0)
print('\\n================ v9_fp8, N_k=16384, batch sweep ================')
_ = sweep('v9_fp8',    [16384], batches=[1, 8, 32, 64, 128], head_dims=[128], h_kvs=[1], max_ws_gb=16.0)

## 8. Optional ncu cross-check — L2 hit-rate + DRAM% (root only; counter-free %HBM is primary)

In [ ]:
# OPTIONAL ncu cross-check (root only; skips cleanly on ERR_NVGPUCTRPERM). Reads L2 hit-rate + DRAM%
# for ONE L2-resident shape and ONE past-L2 shape via the --profile entrypoint. The counter-free %HBM /
# L2! signal above is the PRIMARY verdict; ncu just corroborates.
import subprocess
METRICS = ('lts__t_sector_hit_rate.pct,'
           'dram__throughput.avg.pct_of_peak_sustained_elapsed,'
           'lts__throughput.avg.pct_of_peak_sustained_elapsed')
for tag, shape in (('L2-resident N_k=2048', '1,1,2048,128'), ('past-L2 N_k=65536', '1,1,65536,128')):
    print(f'\\n===== ncu {tag} (v8_gqa_ss) =====')
    cmd = ['ncu', '--metrics', METRICS, '--launch-count', '5',
           '--kernel-name', 'regex:(gqa_ss|fp8)', '--target-processes', 'all',
           sys.executable, '-m', 'bench.regime', '--profile', shape, '--backend', 'v8_gqa_ss']
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        print(out.stdout[-2500:] if out.stdout else '(no stdout)')
        if 'ERR_NVGPUCTRPERM' in (out.stdout + out.stderr):
            print('>>> ncu blocked (ERR_NVGPUCTRPERM): need root/--cap-add. Counter-free %HBM stands.')
    except FileNotFoundError:
        print('>>> ncu not installed on this runtime; skipping (counter-free %HBM is the primary verdict).'); break
    except Exception as e:
        print('>>> ncu failed:', type(e).__name__, e)

## 9. Reset clocks

In [ ]:
from bench.regime import reset_clocks
reset_clocks()

## 10. Verdict (fill after the run)

Read the **%HBM vs N_k** plot + the `L2served` column:

| Outcome | Signature | Means |
|---|---|---|
| **Memory-bound past L2** | %HBM climbs toward ~70% as N_k passes the L2 crossing; `L2!` fires only at small N_k | the decode kernel IS bandwidth-bound once the KV spills L2 → FP8/NVFP4 byte cuts pay off; the 6-step "per-CTA" read was an L2 artifact |
| **Per-CTA-bound (confound-free)** | %HBM stays ~10% even past L2; `L2served=False` throughout; large-batch sweep stays flat | the kernel is genuinely launch/per-CTA-bound; bytes are not the wall here → persistent-kernel / megakernel is the lever, FP8 stays capacity+accuracy |
| **L2-resident (the confound, shown)** | `L2!` fires (eff_bw > HBM peak) while WS ≤ 4 MB | confirms the C12 critique directly — %HBM was never a boundedness metric at those sizes |

Then fill `docs/results.md` Step 9 Task 1 + commit `docs/diagrams/v9-task1-regime.svg`.